In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Create directories
os.makedirs('data', exist_ok=True)
os.makedirs('visuals', exist_ok=True)

def fetch_bdapi_metadata(study_id):
    """
    Uses the NASA BDAPI v2 endpoint to fetch ISA-Tab metadata directly into a DataFrame.
    """
    print(f"Querying BDAPI Metadata for OSD-{study_id}...")
    
    # Using the exact endpoint specified in the OSDR documentation
    url = f"https://visualization.osdr.nasa.gov/biodata/api/v2/query/metadata/?id.accession=OSD-{study_id}"
    
    try:
        # The API natively returns a CSV formatted response, which pandas can read directly from the URL!
        df = pd.read_csv(url)
        
        # Add a helper column so we know which dataset this belongs to when combined
        df['Dataset_ID'] = f"OSD-{study_id}"
        
        # Clean up column names to avoid trailing spaces from the API
        df.columns = [c.strip() for c in df.columns]
        
        print(f" -> Successfully loaded {len(df)} metadata records for OSD-{study_id}.")
        return df
        
    except Exception as e:
        print(f" -> Error fetching OSD-{study_id}: {e}")
        return pd.DataFrame() # Return empty DataFrame on failure

# 1. Fetch the exact metadata for both Gut Microbiome datasets
df_osd249_meta = fetch_bdapi_metadata("249")
df_osd466_meta = fetch_bdapi_metadata("466")

# Safety Check: If both failed, stop execution to prevent KeyError
if df_osd249_meta.empty and df_osd466_meta.empty:
    raise ValueError("The BDAPI returned no data for both datasets. Please check your internet connection or API status.")

# 2. Save strictly named CSV files for your partner
csv_249 = 'data/df_osd249_metadata.csv'
csv_466 = 'data/df_osd466_metadata.csv'

if not df_osd249_meta.empty:
    df_osd249_meta.to_csv(csv_249, index=False)
if not df_osd466_meta.empty:
    df_osd466_meta.to_csv(csv_466, index=False)

print(f"\nSaved Metadata Tables to disk:")
print(f" - {csv_249}")
print(f" - {csv_466}")

# 3. Analyze what Data Types are available to download next
# We will combine them to look at the 'file.datatype' or 'file.datatype' column
df_combined_meta = pd.concat([df_osd249_meta, df_osd466_meta], ignore_index=True)

# The BDAPI docs note it could be 'file.data type' or 'file.datatype'. Let's find which one exists.
datatype_col = 'file.datatype' if 'file.datatype' in df_combined_meta.columns else 'file.data type'

if datatype_col in df_combined_meta.columns:
    type_counts = df_combined_meta.groupby(['Dataset_ID', datatype_col]).size().reset_index(name='Count')
    
    # 4. Create Visualization of Available Data Types
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, ax = plt.subplots(figsize=(10, 6), dpi=300)
    
    sns.barplot(
        data=type_counts, 
        y=datatype_col, 
        x='Count', 
        hue='Dataset_ID', 
        palette=['#1E3A8A', '#0D9488'], # Spaceflight Blue, GC Teal
        ax=ax
    )
    
    ax.set_title('Available Biological Data Types via BDAPI', weight='bold', pad=15)
    ax.set_xlabel('Number of Sample Records', weight='bold')
    ax.set_ylabel('Data Type', weight='bold')
    
    # Add labels to bars
    for container in ax.containers:
        ax.bar_label(container, padding=3, fontsize=9)
        
    plt.tight_layout()
    fig_path = 'visuals/BDAPI_Available_Data_Types.png'
    plt.savefig(fig_path, bbox_inches='tight')
    print(f"\nVisualization saved to {fig_path}")
    
    print("\nAvailable Data Types found:")
    print(type_counts)
    
else:
    print(f"\nNote: The column '{datatype_col}' was not found in the metadata. The datasets might be structured differently.")
    print("Available columns to query:", list(df_combined_meta.columns))

Querying BDAPI Metadata for OSD-249...
 -> Successfully loaded 222 metadata records for OSD-249.
Querying BDAPI Metadata for OSD-466...
 -> Successfully loaded 62 metadata records for OSD-466.

Saved Metadata Tables to disk:
 - data/df_osd249_metadata.csv
 - data/df_osd466_metadata.csv

Note: The column 'file.data type' was not found in the metadata. The datasets might be structured differently.
Available columns to query: ['id.accession', 'id.assay name', 'id.sample name', 'Dataset_ID']


In [1]:
df_osd249_meta

NameError: name 'df_osd249_meta' is not defined

# OSD-466 Extraction

In [10]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style and create output directory
sns.set_theme(style="whitegrid")
OUTPUT_DIR = "OSD-466_Analysis_Outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("[1/4] Loading metadata and filtering WT Flight (FLT) vs Ground Control (GC)...")
meta_df = pd.read_csv('data/df_osd466_metadata.csv')

def parse_sample_name(name):
    parts = str(name).split('_')
    if len(parts) >= 5:
        return {
            'sample_name': name,
            'condition': parts[2],  # FLT, GC, BSL, VIV
            'genotype': parts[3],   # WT, KO
        }
    return None

parsed_meta = pd.DataFrame([parse_sample_name(x) for x in meta_df['id.sample name']])
merged_meta = pd.concat([meta_df, parsed_meta], axis=1)

# Filter: WT only, FLT or GC only (drops BSL, VIV, KO)
target_meta = merged_meta[
    (merged_meta['genotype'].str.upper() == 'WT') & 
    (merged_meta['condition'].isin(['FLT', 'GC']))
].copy()

flt_samples = target_meta[target_meta['condition'] == 'FLT']['id.sample name'].tolist()
gc_samples = target_meta[target_meta['condition'] == 'GC']['id.sample name'].tolist()
target_samples = flt_samples + gc_samples

print(f"--> Confirmed target samples: {len(flt_samples)} FLT, {len(gc_samples)} GC")

print("[2/4] Loading taxonomic abundance table and filtering species-level profiles...")
tsv_path = 'data/GLDS-466_GMetagenomics_Metaphlan-taxonomy_GLmetagenomics.tsv'

if not os.path.exists(tsv_path):
    raise FileNotFoundError(f"Please place '{tsv_path}' in your current working directory.")

# Read MetaPhlAn TSV file, skipping comment lines with comment='#'
abundance_df = pd.read_csv(tsv_path, sep='\t', comment='#')

# Set clade_name as the index
if 'clade_name' in abundance_df.columns:
    abundance_df = abundance_df.set_index('clade_name')

# Filter for species-level taxonomic signatures (|s__)
species_df = abundance_df[abundance_df.index.str.contains(r'\|s__')].copy()

def clean_taxa_name(name):
    parts = name.split('|')
    for p in reversed(parts):
        if p.startswith('s__'):
            return p.replace('s__', '').replace('_', ' ')
    return name

species_df.index = [clean_taxa_name(idx) for idx in species_df.index]

# Filter columns to match our available target samples
available_samples = [s for s in target_samples if s in species_df.columns]
filtered_species = species_df[available_samples]

# Normalize to percentage relative abundance per sample
rel_abundance = filtered_species.div(filtered_species.sum(axis=0), axis=1) * 100

print("[3/4] Computing Top 10 most abundant species per individual and group means...")
ind_records = []
for sample in available_samples:
    group = 'FLT' if sample in flt_samples else 'GC'
    top10 = rel_abundance[sample].sort_values(ascending=False).head(10)
    for taxa, val in top10.items():
        ind_records.append({
            'Sample_ID': sample,
            'Group': group,
            'Taxa': taxa,
            'Relative_Abundance_Pct': val
        })

df_ind_top10 = pd.DataFrame(ind_records)
df_ind_top10.to_csv(f"{OUTPUT_DIR}/OSD-466_Individual_Top10_Abundance.csv", index=False)

# Group means comparison
active_flt = [s for s in flt_samples if s in available_samples]
active_gc = [s for s in gc_samples if s in available_samples]

mean_abundance = pd.DataFrame({
    'FLT_Mean': rel_abundance[active_flt].mean(axis=1),
    'GC_Mean': rel_abundance[active_gc].mean(axis=1)
})

top10_flt = set(mean_abundance['FLT_Mean'].sort_values(ascending=False).head(10).index)
top10_gc = set(mean_abundance['GC_Mean'].sort_values(ascending=False).head(10).index)
combined_top = list(top10_flt.union(top10_gc))

df_group_comp = mean_abundance.loc[combined_top].reset_index()

# Ensure standard column naming for the taxonomy/taxa identifier
if 'clade_name' in df_group_comp.columns:
    df_group_comp.rename(columns={'clade_name': 'Taxa'}, inplace=True)
elif 'index' in df_group_comp.columns:
    df_group_comp.rename(columns={'index': 'Taxa'}, inplace=True)

df_group_comp.to_csv(f"{OUTPUT_DIR}/OSD-466_Group_Mean_Comparison.csv", index=False)

print("[4/4] Generating visualization charts...")
plt.figure(figsize=(12, 7))

# Dynamic check and identifier resolution for melting
if 'Taxa' not in df_group_comp.columns:
    df_group_comp = df_group_comp.reset_index()

taxa_col = None
for col in df_group_comp.columns:
    if str(col).lower() in ['taxa', 'taxonomy', 'species', 'genus', 'clade']:
        taxa_col = col
        break

if taxa_col is None:
    taxa_col = df_group_comp.columns[0]

df_melted = df_group_comp.melt(id_vars=taxa_col, var_name='Group', value_name='Mean_Abundance_Pct')

if taxa_col != 'Taxa':
    df_melted = df_melted.rename(columns={taxa_col: 'Taxa'})

df_melted['Group'] = df_melted['Group'].map({'FLT_Mean': 'Flight (FLT)', 'GC_Mean': 'Ground Control (GC)'})

sns.barplot(
    data=df_melted, 
    x='Mean_Abundance_Pct', 
    y='Taxa', 
    hue='Group', 
    palette={'Flight (FLT)': 'darkorange', 'Ground Control (GC)': 'steelblue'}
)
plt.title('Top Dominant Microbial Species: WT Flight vs Ground Control (OSD-466)')
plt.xlabel('Mean Relative Abundance (%)')
plt.ylabel('Species')
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/OSD-466_Group_Comparison_Plot.png", dpi=300)
plt.close()

print(f"Success! All outputs saved cleanly in '{OUTPUT_DIR}/'.")

[1/4] Loading metadata and filtering WT Flight (FLT) vs Ground Control (GC)...
--> Confirmed target samples: 3 FLT, 5 GC
[2/4] Loading taxonomic abundance table and filtering species-level profiles...
[3/4] Computing Top 10 most abundant species per individual and group means...
[4/4] Generating visualization charts...
Success! All outputs saved cleanly in 'OSD-466_Analysis_Outputs/'.
